# Agent Patterns

**Prerequisites:** notebooks 2 through 5 of this chapter

Agentic patterns are usually taught as a list of architectures — ReAct, Plan & Execute, ReWOO, Tree of Thoughts, Reflexion, Debate — each with its own diagram, as though each needed its own system.

They don't. Every one of them is a different arrangement of the same handful of components, and this notebook writes each as a declared loop over what notebooks 2 through 5 already built. Nothing new gets implemented here. That's the point: if a pattern needs new machinery, the decomposition was wrong.

**By the end you'll have:**
- Eleven named patterns written as declared loops, each one a list of components already in `runtime.py`, `observe.py`, `think.py`, and `act.py`
- A clear view of what actually distinguishes them — usually one component, or one flag
- Every loop constructed for real, so none of it is pseudocode
- An honest account of the one thing these components genuinely can't express

## Setup

Every import is something built earlier in this chapter. Nothing is defined in this notebook.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
sys.path.append(str(Path.cwd().parent.parent / "01_foundations"))

from chat import Conversation
from runtime import Runtime, Loop, Graph
from observe import Observe
from think import Think, PlannerThink, SelectThink, ExpandThink, ReflectionThink, Fork
from act import Act

# every pattern below, as a factory returning a declared Runtime
from patterns import (react, plan_and_execute, rewoo, llm_compiler, self_refine, reflexion,
                      self_consistency, debate, tree_of_thoughts, graph_of_thoughts)

In [2]:
from datetime import datetime
from typing import Literal
from zoneinfo import ZoneInfo, ZoneInfoNotFoundError

from pydantic import BaseModel, field_validator

# Real tools, because these patterns are run against a real model at the end of the notebook.
# A stand-in tool that "does a thing" gives it nothing to reason about.
def get_current_time(timezone):
    return datetime.now(ZoneInfo(timezone)).strftime("%Y-%m-%d %H:%M %Z")

def convert_temperature(value, from_unit, to_unit):
    if from_unit.upper() == "F" and to_unit.upper() == "C":
        return round((value - 32) * 5 / 9, 1)
    if from_unit.upper() == "C" and to_unit.upper() == "F":
        return round(value * 9 / 5 + 32, 1)
    raise ValueError(f"unsupported conversion: {from_unit} to {to_unit}")

class GetCurrentTimeArgs(BaseModel):
    timezone: str

    @field_validator("timezone")
    @classmethod
    def timezone_must_be_valid(cls, v):
        try:
            ZoneInfo(v)
        except ZoneInfoNotFoundError:
            raise ValueError(f"'{v}' is not a valid IANA timezone")
        return v

class ConvertTemperatureArgs(BaseModel):
    value: float
    from_unit: Literal["C", "F"]
    to_unit: Literal["C", "F"]

FUNCTIONS = {"get_current_time": get_current_time, "convert_temperature": convert_temperature}
SCHEMAS = {"get_current_time": GetCurrentTimeArgs, "convert_temperature": ConvertTemperatureArgs}
OUTPUTS = {"get_current_time": "a date and time string",
           "convert_temperature": "a single number"}
TOOLS = [
    {"type": "function", "function": {
        "name": "get_current_time",
        "description": "Get the current date and time in a given IANA timezone.",
        "parameters": {"type": "object",
                       "properties": {"timezone": {"type": "string", "description": "IANA timezone name"}},
                       "required": ["timezone"]}}},
    {"type": "function", "function": {
        "name": "convert_temperature",
        "description": "Convert a temperature value between Celsius and Fahrenheit.",
        "parameters": {"type": "object",
                       "properties": {"value": {"type": "number"},
                                      "from_unit": {"type": "string", "enum": ["C", "F"]},
                                      "to_unit": {"type": "string", "enum": ["C", "F"]}},
                       "required": ["value", "from_unit", "to_unit"]}}},
]

# Patterns are not interchangeable, so one request cannot suit all ten. The acting patterns get
# something with tool calls in it; the ones that generate and compare ideas get something to
# actually have opinions about.
TOOL_REQUEST = "What's the current time in Tokyo, and what's 100 degrees Fahrenheit in Celsius?"
IDEA_REQUEST = "Suggest a memorable name for a running-shoe brand aimed at beginners."
DEBATE_REQUEST = "Should a small startup write its backend in Python or in Go?"

COMMON = dict(tools=TOOLS, functions=FUNCTIONS, schemas=SCHEMAS)

def describe(name, runtime):
    parts = []
    for c in runtime.loop:
        if isinstance(c, Loop):
            inner = " -> ".join(type(x).__name__ for x in c.components)
            stop = f"count={c.count}" if c.count is not None else f"until={c.until.__name__}"
            parts.append(f"Loop[{inner}, {stop}]")
        else:
            parts.append(type(c).__name__)
    tail = "" if runtime.repeat_from is None else f"   (repeat_from={runtime.repeat_from})"
    print(f"{name:18} {' -> '.join(parts)}{tail}")

PATTERNS = {}

## The primitives, one more time

Four kinds of thing, and everything below is built from them.

| | what it does | never does |
|---|---|---|
| **`Observe`** | brings something in — a user event, or a raw effect turned into a readable observation | decide what happens next |
| **`Think`** | reasons over the conversation and produces a decision, plan, or critique | change anything outside the program |
| **`Act`** | carries out a decision already made | decide anything |
| **`Runtime` / `Loop` / `Graph`** | schedule which component runs when | reason |

The Thinks differ only in what they're asked and what shape they return: `Think` decides a tool call, a delegation, or an answer; `PlannerThink` writes a whole sequence; `ReflectionThink` critiques without answering; `SelectThink` scores candidates and prunes.

## Why Chain of Thought isn't here

It is the pattern everyone expects in a list like this, and it is missing on purpose.

Chain of Thought changes what the model produces **inside a single call** — ask it to reason step by step and it writes its working before its answer. Nothing about the arrangement of calls changes. A declared loop can only express *how calls are ordered*, so there is no loop that is Chain of Thought and no loop that isn't.

An earlier draft of this notebook did list it, as `Observe -> Think -> Act`. That declaration contains no chain-of-thought prompt at all; it is simply one model call with a famous name attached, which is worse than omitting it.

Prompting strategies live in `01_foundations/04_prompting.ipynb`. Everything below is a shape.

Every other entry survives that test. Self-Consistency and Debate look like prompting tricks but genuinely need several independent runs and a join, which is structure.


## ReAct

Interleave reasoning and acting: decide one step, do it, look at the result, decide again.

```python
Runtime(
    loop=[
        Observe(event=request),
        Think(tools=tools, allow_fork=fork),
        Act(functions=functions, schemas=schemas, branch_runtime=...),
        Observe(),
    ],
    repeat_from=1,
)
```

Four components and a `repeat_from`. Notebook 2 built this before the word "ReAct" appeared.

`fork=True` (the default) lets the `Think` split the request into sub-agents when it judges that worth the cost. That is **not a separate pattern** — it is this loop with delegation switched on, and the shape does not change either way. `Think` withholds `delegate` for exactly one turn after branch results come back, which is what removed the near-duplicate second `Think` these loops used to carry.

In [3]:
# fork=False: plain ReAct. Delegation is available here (see notebook 5) but a two-lookup
# request is precisely the case the guidance says NOT to delegate -- six tool calls you can
# make yourself beat standing up two sub-agents to make them for you.
PATTERNS["ReAct"] = react(TOOL_REQUEST, fork=False, max_iterations=12, **COMMON)

## Plan & Execute

Decide what needs achieving, then work out how — one goal at a time, looking at results as you go.

```
Observe(event) -> PlannerThink(goals=True) -> [ Think -> Act -> Observe ]  repeating
```

The planner writes **goals**, not calls: *"find the current time in Tokyo"*, not `get_current_time(timezone="Asia/Tokyo")`. It names no tool and emits no decision, because working out how to reach a goal is reasoning. The `Think` after it does that, reading each result before choosing the next call.

That is the whole of what separates this from the two patterns below. Notebook 4 builds it.

**Run live in notebook 4.**


**A footnote on `replan`, learned the hard way.** Its first version explained in its own description when to revise a plan and how, and embedded the current goals — 1202 characters, twice the size of both real tools combined. Offered alongside them, llama-3.3-70b (the model this was measured on) then rejected *every* attempt to call *any* tool, five times running, emitting llama's `<function=name{...}</function>` text form instead of a real tool call. Remove `replan` and the identical request succeeded on the first try.

The fix was the lesson notebook 5 already records about `delegate`: prose about how to use a tool belongs in the system message, where it costs the generator nothing. The schema is now 40 characters of description and two fields, and the guidance reads the same. What changed is where it sits.


In [4]:
# fork=False for the same reason as ReAct: these goals are two lookups, and delegating work you
# could finish in a couple of calls is what the guidance says not to do. `replan` stays on -- the
# planner runs once and is never re-entered, so this Think is the only thing that can revise the
# goals if executing them turns something up.
PATTERNS["Plan & Execute"] = plan_and_execute(TOOL_REQUEST, outputs=OUTPUTS, fork=False,
                                              max_iterations=12, **COMMON)

## ReWOO and LLM Compiler

Both plan **exact calls** up front — not goals — writing `$stepN` for any argument that cannot be known yet. `Graph` then runs them, starting each the moment its own references are satisfied, and **nothing consults the model in between**. That shared idea is what ReWOO's name states: *Reasoning WithOut Observation*. Against ReAct, which re-sends a growing conversation every step, the model is called once to plan and once at the end, however many steps there are.

They share a planner and they share workers. What differs is the **third role**.

| | third role | can it ask for more work? |
|---|---|---|
| **ReWOO** | a **solver** — a plain `Think` that reads the plan and the evidence and writes the answer | no |
| **LLM Compiler** | a **joiner** — `PlannerThink` again, now running `ContinueOrFinish` | yes, and `repeat_from` gives it somewhere to go |

That is the whole difference, and two things follow from it.

**ReWOO terminates by construction.** Its solver can only answer, so the run ends. Give it a joiner instead and you have a component that may reply "not done, here are more steps" — in a loop with nowhere to put them, which ends with tool output and no answer at all.

**ReWOO costs no less.** Its solver and the compiler's joiner are the same single call in the same position; one of them simply has an extra option. So ReWOO is a one-round LLM Compiler forbidden from noticing it isn't finished: identical price, strictly fewer choices.

Which makes ReWOO worth *recognising* rather than choosing. Its contribution was historical — in 2023, decoupling reasoning from observation was the argument that mattered against ReAct's cost. LLM Compiler kept the idea and dropped the limitation.

Parallelism, incidentally, is **not** the difference, though the usual summaries say so. ReWOO's plan carries the same `$stepN` information, so `Graph` schedules it identically. Both are parallel here.

**The compiler loop runs live in notebook 4**, which is where the faults in it were found. ReWOO is the same planner and workers with a solver on the end, and is not separately run.


**The solver holds no tools, and that is what makes it a solver.** Its job is to answer from the evidence `Graph` already gathered. Handed the tools again it simply calls them again — run live, it re-fetched a time the plan had already fetched, three turns running, until the budget stopped it. Withholding the tools is not cornering it into an answer; by that point answering is genuinely the only thing left to decide.


In [5]:
PATTERNS["ReWOO"] = rewoo(TOOL_REQUEST, outputs=OUTPUTS, **COMMON)

The compiler needs no trailing `Act` because `Graph` handles the terminal case itself: when the newest decision is a `respond`, it runs a graph of exactly one Act — the one that emits the answer. ReWOO's `Act` is explicit instead, which reads more plainly for a loop that only ever goes round once.


In [6]:
PATTERNS["LLM Compiler"] = llm_compiler(TOOL_REQUEST, outputs=OUTPUTS,
                                       max_iterations=12, **COMMON)

## Self-Refine

Draft, critique the draft, redraft. One agent, no tools — the critique judges *prose*, so nothing executes.

```
Observe(event) -> Loop[ Think -> ReflectionThink, count=REFINE_ROUNDS ] -> Think -> Act
```

`Think` produces a `respond` decision, which is only an intention — nothing ends a run until an `Act` converts one into an `assistant` message. So drafts accumulate inside the `Loop` and are critiqued, and `Act` runs once at the end on the draft that survived.

The obvious alternative, `Observe -> Think -> Act -> ReflectionThink` cycled with `repeat_from`, never reflects at all: `Act` converts the first draft and `Runtime` stops on seeing an `assistant` message.

`count` is a parameter rather than a constant because how much refining a task deserves is the caller's judgement.

**Run live in notebook 3.**


In [7]:
PATTERNS["Self-Refine"] = self_refine(IDEA_REQUEST, refine_rounds=2, **COMMON)

## Reflexion

Self-Refine with tool use: make an attempt over several turns, then critique the whole attempt rather than a single answer.

```
Observe(event) -> Loop[ Think -> Act -> Observe, count=ATTEMPT_TURNS ] -> ReflectionThink   repeating
```

`Act` and `Observe` are inside the cycle here, and that is the difference from Self-Refine: the critique judges *work*, so the work has to actually run and be reported before there is anything to judge. `count` decides how much work counts as one attempt — the gap between "that tool call was wrong" and "that approach was wrong".

`repeat_from=1` re-enters the same `Loop`, so the next attempt is made with the critique in hand.

**Run live in notebook 3.**


In [8]:
PATTERNS["Reflexion"] = reflexion(TOOL_REQUEST, attempt_turns=3,
                                 max_iterations=12, **COMMON)

## Self-Consistency

Answer the same question several times with different sampling, then reconcile the answers.

```python
Runtime(loop=[
    Observe(event=request),
    Fork(branches=[{"goal": request, "temperature": t} for t in temperatures]),
    Act(functions=functions, schemas=schemas, branch_runtime=...),
    Observe(),                      # joins every branch's answer into one message
    Think(tools=tools, allow_fork=False),
    Act(functions=functions, schemas=schemas),
])
```

**The fork is declared, not asked for.** An earlier version wrote a paragraph telling the model to *"delegate this exact question to 3 branches... do not change the wording between branches"* — configuration written in English and parsed by a model that, when this was written, emitted `delegate` malformed about a quarter of the time. It had to *plead* for identical wording; one list comprehension guarantees it.

That is what `Fork` is: the same decision `Think` produces for `delegate`, written by the caller instead, so `Act` needs no new code to handle it. The precedent is `Observe(event=...)` — a component handed fixed content that emits it rather than computing one.

**There is no voting component.** `Observe` merges every branch's answer into one labelled message, and a `Think` reading three labelled answers and picking the recurring one is just a `Think` reading its context. A `VoteThink` would hardcode majority, when "two agree but the third reasons better" is worth keeping available.

In [9]:
PATTERNS["Self-Consistency"] = self_consistency(IDEA_REQUEST, temperatures=(0.2, 0.8, 1.2),
                                                **COMMON)

## Debate

Several positions argued out, each branch keeping its own history and hearing the others' turns.

```python
Runtime(loop=[
    Observe(event=request),
    Fork(branches=sides, persistent=True, broadcast=True, start=sides[0]["id"]),
    Act(functions=functions, schemas=schemas, branch_runtime=...),
    Observe(),
    Loop([Think(tools=None, allow_fork=False, allow_advance=True),   # who speaks next
          Act(functions=functions, schemas=schemas, branch_runtime=...),
          Observe()],
         count=turns),
    Think(tools=None, allow_fork=False, allow_advance=False),        # sum up; nothing else left
    Act(functions=functions, schemas=schemas),
])
```

Same machinery as Self-Consistency plus turn-taking; the difference is entirely in the flags.

**`persistent`** keeps each branch's conversation alive on `conversation.branches`. **`broadcast`** copies each turn into every sibling's history, attributed to whoever said it. **`start`** names who opens — without it every side speaks at once and nobody is responding to anything, which is not a debate.

**`allow_advance=True` with `allow_fork=False`** is the pair that makes the moderator work, and those used to be a single flag. It should choose *which* branch continues and never start a fresh one, because a new branch begins with none of what the existing one established. While the two were welded together, turning delegation off took turn-taking with it and this component could not be expressed at all.

**Neither `Think` here holds tools.** A moderator's job is to pick who speaks and to say when the exchange has settled; looking things up is the branches' work. It is the same reasoning as ReWOO's solver — and in this notebook the tools are a clock and a temperature converter, which have nothing to contribute to a debate about language choice. Every schema a model holds is another chance for it to emit a malformed call instead of prose.

### Why the exchange is a `Loop` rather than a `repeat_from`

An exchange has no natural end. Cycled with `repeat_from`, this moderator advanced a branch on every single turn until the budget stopped it, and the run finished on `Stopping: reached the maximum of 18 iterations` rather than on a conclusion.

Adding *"once the exchange has covered what matters, stop advancing and answer instead"* to the system message did not fix it. Asked to wrap up, it advanced a branch to do the wrapping up — its last decision was `Continuing [python]: Summarize the main points of the discussion so far.` It delegated the conclusion.

So the caller says how many turns, exactly as Self-Refine and Reflexion do, and the `Think` that follows the loop has **no `advance` tool and no tools at all**. Summing up is the only move it has left. Telling a model when to stop is a suggestion; taking the tool away is a fact.

With two sides, alternating is obvious and turn-taking buys little. With three or more it is the whole problem: somebody has to judge who is worth hearing from next, and judging is what `Think` is for.

In [10]:
SIDES = [{"id": "python", "goal": f"{DEBATE_REQUEST} Argue the case for Python."},
         {"id": "go", "goal": f"{DEBATE_REQUEST} Argue the case for Go."}]

# No tools, and not to dodge anything: this debate is about language choice, and a clock and a
# temperature converter have nothing to contribute to it. Passing them anyway gives each debater
# two schemas it will never legitimately call, and every schema a model holds is a chance for it
# to emit a malformed call instead of prose. Branches CAN have tools -- a debater checking a fact
# is a fine thing -- but only tools its argument could actually use.
#
# max_iterations=18: opening the debate costs four (Observe, Fork, Act, Observe) and each exchange
# after it costs three, so ten leaves room for two turns -- not enough for either side to answer
# the other and for the moderator to judge it settled.
PATTERNS["Debate"] = debate(DEBATE_REQUEST, SIDES, start="python",
                            tools=None, functions=None, schemas=None,
                            max_iterations=18)

## Tree of Thoughts

Explore several lines of thinking at once, score them, keep the best few, and grow those.

```python
Runtime(
    loop=[
        Observe(event=request),
        ExpandThink(thoughts=2, max_parents=1),     # <- 1 makes it a tree
        Act(functions=functions, schemas=schemas, branch_runtime=...),
        SelectThink(keep=2),
        Act(functions=functions, schemas=schemas),
        Observe(),
        Think(tools=tools, allow_fork=False),      # answer, or go round again
        Act(functions=functions, schemas=schemas),
    ],
    repeat_from=1,
)
```

`thoughts` is the branching factor, `keep` the beam width. Both belong to the caller, exactly like `refine_rounds` — change them and it is still Tree of Thoughts.

**Which parent a new thought belongs to is not a decision.** `ExpandThink` asks the model once per surviving branch, showing that branch's working and nothing else, so every thought grows out of that line by construction. An earlier version had the model name the parent alongside the thought, which is a choice masquerading as a fact: it could write a thought inspired by one line and label it another, and nothing could catch that, because only the model knew where the thought came from. Asking per parent makes a wrong parent impossible rather than merely unlikely — and it is what the paper does, handing its generator one state.

**A child starts from its parent's working, compacted.** Not the parent's raw messages, which would drag every intermediate turn along, but the block `transcript()` renders: goal, calls, why, results, answer. That block *is* the inherited state, which is why a search branch runs with `explain=True` — strip the reasons out and the most useful part of the inheritance goes with them.

**Backtracking works, and needed one thing.** Pruning only ever filtered `pending`, so an abandoned branch was always still there — but `SelectThink` scored a round and then forgot the scores, so nothing knew *which* abandoned line was worth returning to. Scores are now recorded against branch ids, which outlive the round: a line dropped in round one at 6/10 still beats everything round three produced, and is still reachable.

**A search needs a way out.** `ExpandThink` always expands and `SelectThink` always prunes; neither can ever decide the answer is good enough. Without the `Think` at the end the loop cycles until the budget stops it and every run ends on `Stopping:` — which is exactly what happened the first time this notebook was executed against the real model. Deciding when to stop searching is a judgment, and the search itself cannot make it.

In [11]:
PATTERNS["Tree of Thoughts"] = tree_of_thoughts(IDEA_REQUEST, thoughts=2, keep=2,
                                                max_iterations=20, **COMMON)

## Graph of Thoughts

Tree of Thoughts plus the one move a tree cannot make: **merging** several thoughts into one.

```python
Runtime(
    loop=[
        Observe(event=request),
        ExpandThink(thoughts=2, max_parents=2),     # <- the only difference
        Act(functions=functions, schemas=schemas, branch_runtime=...),
        SelectThink(keep=3),
        Act(functions=functions, schemas=schemas),
        Observe(),
        Think(tools=tools, allow_fork=False),      # answer, or go round again
        Act(functions=functions, schemas=schemas),
    ],
    repeat_from=1,
)
```

The component list is identical to Tree of Thoughts, and that is the point rather than laziness. They differ by **one number** — the same way Self-Consistency and Debate differ only by flags.

A tree node has one parent, so a tree can only ever split. `max_parents=1` means no merge is ever offered, so the shape *cannot* become a graph. Above 1, each round asks one extra question — *"is any group of these worth combining into a single stronger idea?"* — and a yes produces a node with several parents. That is the aggregation edge.

**Merging is the one thing that cannot be structural.** Growth is deterministic: one parent, settled by which line the model was shown. But nothing about the loop can work out which two ideas are worth joining, so that stays a real decision — which is how the paper divides it too, with `Generate` taking one thought and only `Aggregate` taking several.

An earlier version of this notebook said the difference was *persistence*: that Tree of Thoughts discards its branches and so cannot backtrack. That was wrong twice over. The paper keeps its whole tree, and its depth-first variant returns to abandoned lines; and in our code a declared loop does not decide persistence at all.

In [12]:
PATTERNS["Graph of Thoughts"] = graph_of_thoughts(IDEA_REQUEST, thoughts=2, keep=3,
                                                  max_parents=2, max_iterations=20,
                                                  **COMMON)

## All ten

Every loop above is a real object built from real classes. Printing their component sequences is the check that none of this is pseudocode — a name that didn't exist, or an argument that wasn't accepted, would have raised on construction.

Construction is a weak check, though, and it is worth being honest about how weak. It proves the names resolve and the arguments fit. It cannot tell you a loop terminates. An earlier version of the LLM Compiler entry below was missing the component that turns a decision into an answer, so it would have planned, executed, concluded it was done, and then re-run the finished plan until its budget stopped it — and it constructed perfectly happily. What caught it was reasoning through a run by hand, not building the object.

So the table each pattern ends with names **where it was actually executed**. That is the real evidence; this cell only checks the declarations are well formed.


In [13]:
for name, runtime in PATTERNS.items():
    describe(name, runtime)

ReAct              Observe -> Think -> Act -> Observe   (repeat_from=1)
Plan & Execute     Observe -> PlannerThink -> Think -> Act -> Observe   (repeat_from=2)
ReWOO              Observe -> PlannerThink -> Graph -> Think -> Act -> Observe   (repeat_from=3)
LLM Compiler       Observe -> PlannerThink -> Graph   (repeat_from=1)
Self-Refine        Observe -> Loop[Think -> ReflectionThink, count=2] -> Think -> Act -> Observe   (repeat_from=2)
Reflexion          Observe -> Loop[Think -> Act -> Observe, count=3] -> ReflectionThink   (repeat_from=1)
Self-Consistency   Observe -> Fork -> Act -> Observe -> Think -> Act -> Observe   (repeat_from=4)
Debate             Observe -> Fork -> Act -> Observe -> Loop[Think -> Act -> Observe, count=2] -> Think -> Act
Tree of Thoughts   Observe -> ExpandThink -> Act -> SelectThink -> Act -> Observe -> Think -> Act   (repeat_from=1)
Graph of Thoughts  Observe -> ExpandThink -> Act -> SelectThink -> Act -> Observe -> Think -> Act   (repeat_from=1)


### Actually running them

A declaration that constructs cleanly proves nothing about whether it runs. Every pattern below
is run against the real model, with a spy wrapped round each component counting how often it
actually ran. A pattern passes only if it terminates, ends on a real answer without the runtime having to
stop it, and used every component it declared -- a component that never ran is one
the declaration only *claimed* to use.

That check has earned its place. Running these for the first time turned up six faults that
construction could not have shown: two search loops with no way to ever stop searching, a solver
`Act` declared without the tools it needed, and three patterns that could finish a run with no
answer in it at all.

**"Stopped" is read from `conversation.stopped`, not from the text.** A stopped run used to end
on *"Stopping: reached the maximum…"*, and this check once looked for exactly that. It now ends
on the model's best answer from what it had gathered (notebook 2), which reads like a real one --
so a check on the text would pass a pattern that never finished.

**Each pattern gets its own cell.** Running ten patterns against a real API is not free, and a
single loop over all of them means fixing one costs a re-run of all ten.


In [14]:
import time

class Spy:
    """Wraps a component and counts how often it runs, without changing what it does."""
    def __init__(self, inner, log):
        self.inner, self.log = inner, log
        log.setdefault(type(inner).__name__, 0)
    def run(self, conversation):
        self.log[type(self.inner).__name__] += 1
        return self.inner.run(conversation)

def watch(components, log):
    for c in components:
        if isinstance(c, Loop):
            c.components = watch(c.components, log)
    return [Spy(c, log) for c in components]

# Debate ends on the exchange itself, not on a verdict handed down by the outer loop.
NO_ANSWER_EXPECTED = {"Debate"}
# Reflexion only reflects when an attempt fails to finish the job. Given a request its first
# attempt can complete, Act turns that answer into an assistant message and the run ends before
# ReflectionThink is reached -- the design working, not a component going unused.
MAY_SKIP = {"Reflexion": {"ReflectionThink"}}

def check(name):
    """Run one pattern, report whether it held up, and show what it said."""
    rt = PATTERNS[name]
    log = {}
    rt.loop = watch(rt.loop, log)
    started = time.monotonic()
    try:
        convo = rt.run(Conversation())
    except Exception as e:
        body = getattr(e, "body", None)
        detail = (body or {}).get("error", {}).get("message") if isinstance(body, dict) else None
        print(f"FAIL  {name}: raised {type(e).__name__}")
        print(f"      {detail or str(e)}")
        return None
    elapsed = time.monotonic() - started
    last = convo.messages[-1] if convo.messages else {}
    answer = str(last.get("content", ""))

    problems, notes = [], []
    if convo.stopped:
        # not the text: a stopped run ends on a best-effort answer that reads like a real one
        problems.append(f"stopped by the runtime ({convo.stopped}): {answer[:70]!r}")
    elif last.get("role") != "assistant" and name not in NO_ANSWER_EXPECTED:
        problems.append(f"ended on {last.get('role')}, not an answer")
    for k, v in log.items():
        if v == 0:
            (notes if k in MAY_SKIP.get(name, set()) else problems).append(f"never ran: {k}")

    print(f"{'ok  ' if not problems else 'FAIL'}  {name}   {len(convo.messages)} msgs, "
          f"{elapsed:.1f}s   " + ", ".join(f"{k}x{v}" for k, v in log.items()))
    for p in problems:
        print(f"      -> {p}")
    for n in notes:
        print(f"      (expected) {n}")
    print(f"\n      {answer[:300]}")
    return convo

In [15]:
check('ReAct')

ok    ReAct   7 msgs, 3.7s   Observex3, Thinkx3, Actx3

      The current time in Tokyo is 2026‑08‑18 06:39 JST, and 100 °F is 37.8 °C.


In [16]:
check('Plan & Execute')

ok    Plan & Execute   8 msgs, 3.2s   Observex3, PlannerThinkx1, Thinkx3, Actx3

      The current time in Tokyo is 06:40 JST on 2026‑08‑18, and 100 °F is 37.8 °C.


In [17]:
check('ReWOO')

ok    ReWOO   6 msgs, 3.3s   Observex1, PlannerThinkx1, Graphx1, Thinkx1, Actx1

      It is 2026‑08‑18 06:40 JST in Tokyo, and 100 °F equals 37.8 °C.


In [18]:
check('LLM Compiler')

ok    LLM Compiler   6 msgs, 3.3s   Observex1, PlannerThinkx2, Graphx2

      The current time in Tokyo is 2026-08-18 06:41 JST, and 100 °F is equal to 37.8 °C.


In [19]:
check('Self-Refine')

ok    Self-Refine   7 msgs, 4.4s   Thinkx3, ReflectionThinkx2, Observex1, Loopx1, Actx1

      **StrideStart** – a short, catchy name that blends “stride” (running) with “start,” clearly signaling a brand made for beginners taking their first steps into running.


In [15]:
check('Reflexion')

ok    Reflexion   7 msgs, 5.0s   Thinkx3, Actx3, Observex3, Loopx1, ReflectionThinkx0
      (expected) never ran: ReflectionThink

      The current time in Tokyo is 06:42 JST on 2026‑08‑18, and 100 °F equals ≈ 37.8 °C.


In [17]:
check('Self-Consistency')

ok    Self-Consistency   5 msgs, 2.2s   Observex2, Forkx1, Actx2, Thinkx1

      **FirstStride** – a simple, memorable name that emphasizes the very first step in a runner’s journey, making it feel welcoming and supportive for beginners.


In [18]:
check('Debate')

ok    Debate   7 msgs, 24.3s   Thinkx2, Actx3, Observex3, Forkx1, Loopx1

      **Python vs. Go for a Small Startup’s Backend – A Pragmatic Comparison**

| Dimension | Python | Go |
|-----------|--------|----|
| **Time‑to‑market / Development speed** | • Concise, readable syntax and dynamic typing let you prototype instantly.<br>• REPL & interactive notebooks make experiments f


In [15]:
check('Tree of Thoughts')

ok    Tree of Thoughts   6 msgs, 7.0s   Observex2, ExpandThinkx1, Actx3, SelectThinkx1, Thinkx1

      RunEase is the most memorable and clearly conveys effortless progress for beginner runners.


In [16]:
check('Graph of Thoughts')

ok    Graph of Thoughts   6 msgs, 5.5s   Observex2, ExpandThinkx1, Actx3, SelectThinkx1, Thinkx1

      EasyStride


Every pattern terminates, answers, and exercises each component it declared — against the real
model, not a stand-in.

The timings are worth reading alongside the shapes. The forking patterns are not slower in
proportion to their branches, because branches run concurrently; and `Graph` finishes a plan in
roughly the time of its slowest step rather than the sum of them.

What the runs do *not* prove is that a pattern suits its request. Tree of Thoughts will happily
explore a two-step lookup, and Plan & Execute will plan a question it could have answered in one
call. Choosing the pattern is still yours.


Read down that list and the family resemblances are hard to miss.

**ReWOO and LLM Compiler** are the same three components. One is the loop unrolled to two rounds, the other is the loop. Same cost, and only one of them can notice it isn't finished.

**Plan & Execute** shares their planner and nothing else. It is the only one of the three with a `Think` inside its cycle, because its plan holds goals rather than calls and somebody has to work out how.

**Self-Refine and Reflexion** are the same shape; Reflexion wraps the attempt in a `Loop` so the critique lands on a whole attempt rather than one turn.

**Self-Consistency, Debate, Tree of Thoughts and Graph of Thoughts** all fork. Self-Consistency and Debate print an *identical* component list and differ only in the `persistent` and `broadcast` flags on the decision — which means the difference between them isn't in the code at all; it is in what a `Think` decided at runtime. Tree of Thoughts adds `SelectThink`; Graph of Thoughts adds persistence to that.

So they aren't ten architectures. They're a few real choices: whether to plan ahead and whether the plan holds goals or calls, whether to loop, whether to split, whether the splits persist and can see each other, and whether the results get filtered.


## What this can't do

The two limits listed here for most of this chapter turned out to be **the same missing piece**, and it is now built. Both are worth keeping on the page, because how they collapsed into one is more instructive than either was alone.

**Deciding whose turn is next.** `Act` could always resume a branch; nothing could ever decide to. With two debaters alternation is obvious, but past that, choosing who speaks next is judgment — and the obvious implementation, a round-robin inside `Act`, would have put a policy decision in the component that isn't allowed to make any. It is now an `advance` tool on `Think`, offered once persistent branches exist, in exactly the way `delegate` is.

**Backtracking to a discarded candidate.** This looked like a separate, structural limit: `SelectThink` prunes, the losers vanish, Tree of Thoughts can never reconsider one. Following it through, that was wrong. `Act._fork` stores **every** branch on `conversation.branches` when `persistent=True`, and pruning only ever filters `pending` — the results. A discarded candidate's whole conversation was still sitting there, alive and addressable. Nothing had been thrown away; there was simply no way to say "go back to that one". The same `advance` tool says it.

So the limit was never bookkeeping. It was a missing decision, and it was the same missing decision as turn-taking.

What remains true is narrower, and it is a choice rather than a limit: **reopening a branch needs it to have been forked with `persistent=True`.** Without that, branch conversations are discarded once they report and a pruned candidate really is gone. Nothing in a declared loop sets this — `persistent` is a field on the fork decision, so the `Think` chooses it, the same way Self-Consistency and Debate differ only by flags decided at runtime.

**What is still genuinely missing is the search itself.** `SelectThink` scores a round of candidates and then forgets the scores. A real tree or graph search keeps values on its nodes, so it knows which abandoned branch is the most promising to return to. Here, reopening one is possible; knowing *which* one to reopen has nothing to go on but what is still visible in the conversation. That is the honest remaining gap, and it is a smaller one than "cannot backtrack".

### And a limit of this notebook

Building a `Runtime` proves its component names resolve and its arguments fit. It cannot prove the loop terminates, produces an answer, or reaches every component it declares.

Three declarations here were wrong until each was actually run, and all three constructed without complaint:

- **LLM Compiler** had nothing to turn the planner's final decision into an answer. It planned, executed, concluded it was done, then re-ran the finished plan until its budget stopped it.
- **Self-Refine** put `Act` inside the cycle, so the first draft became an `assistant` message and `Runtime` returned before `ReflectionThink` ran even once — a refine loop that could not refine.
- **This notebook never imported `Conversation`**, which nothing noticed while nothing ever ran.

A fourth fault — `Graph` resolving `$stepN` against a fresh per-round dictionary, so a replanned step referring back to an earlier round read the wrong value — could only appear once a graph ran twice, which no demo had done.

That is why every pattern above names the notebook it was executed in, and why the cell below runs all ten against a stub rather than merely building them.


## What you built

Nothing. That's the result.

✓ Wrote eleven named patterns as declared loops over `Runtime`, `Loop`, `Graph`, `Observe`, the four Thinks, and `Act` — with no component added, and no `runtime.py` change between any two of them

✓ Constructed every one for real, so the declarations are verified code rather than diagrams

✓ Located what actually separates them: `repeat_from` for Chain of Thought vs. ReAct; `Loop` vs. `Graph` for Plan & Execute vs. ReWOO; two boolean flags — decided at runtime, not written in the loop — for Self-Consistency vs. Debate

✓ Closed the two limits this notebook used to list — turn-taking and backtracking — by finding they were the same missing decision, and named what genuinely remains: `SelectThink` keeps no scores, so a search has nothing to tell it which abandoned branch to return to

This closes `02_agent_runtime`. The chapter began with one fused function from chapter 1 that could run exactly one agent, and ends with eleven patterns that are all the same runtime walking a different list.